# DATA 205 Final Project: Ingestion and Cleaning
**Student:** Charlene Stephia Mafok  
**Course:** DATA 205, Montgomery College, Fall 2026  
**Project:** Catching the Warning Signs Early: How Depression and Anxiety Are Linked to Suicidal Thoughts in Young People Ages 12 to 25


## Import libraries

In [57]:
import os
import pandas as pd
import numpy as np

In [58]:

CLEAN = "clean_data/"
os.makedirs(CLEAN, exist_ok=True)

# Part 1: Ingestion

## Unzip the NSDUH file

In [59]:
!unzip -o Mafok_NSDUH_2024_Full_csv.zip

unzip:  cannot find or open Mafok_NSDUH_2024_Full_csv.zip, Mafok_NSDUH_2024_Full_csv.zip.zip or Mafok_NSDUH_2024_Full_csv.zip.ZIP.


## Dataset 1: NSDUH 2024

In [60]:
columns = [
    "QUESTID2",                    # person ID
    "ANALWT2_C",                   # survey weight
    "CATAG6",                      # age category
    "AGE3",                        # smaller age ranges
    "IRSEX",                       # sex
    "NEWRACE2",                    # race/ethnicity
    "COUTYP4",                     # county type
    "POVERTY3",                    # family income vs. poverty line
    "HEALTH",                      # overall health
    "YMDEYR", "AMDEYR",            # depression past year (teen, adult)
    "YUSUITHKYR", "IRSUICTHNK",    # serious suicidal thoughts past year (teen, adult)
    "YUSUIPLNYR", "IRSUIPLANYR",   # suicide plan past year (teen, adult)
    "YGADSEVSYMP", "AGADSEVSYMP",  # anxiety level 1-4 (teen, adult)
    "YGADTOTSC", "AGADTOTSC",      # anxiety score 0-21 (teen, adult)
    "YMDETXRX", "AMDETXRX",        # treatment for depression (teen, adult)
]

nsduh_raw = pd.read_csv("Mafok_NSDUH_2024_Full.csv", usecols=columns)
print("Rows:", nsduh_raw.shape[0])
print("Columns:", nsduh_raw.shape[1])

Rows: 58633
Columns: 21


In [61]:
nsduh_raw.head()

,QUESTID2,ANALWT2_C,COUTYP4,AGE3,HEALTH,IRSEX,CATAG6,NEWRACE2,POVERTY3,YGADTOTSC,...,YUSUITHKYR,YUSUIPLNYR,IRSUICTHNK,IRSUIPLANYR,AGADTOTSC,AGADSEVSYMP,AMDEYR,AMDETXRX,YMDEYR,YMDETXRX
0,10000616,1922.385971,1,7,1,2,3,1,3.0,NaN,...,NaN,NaN,0.0,0.0,0.0,1.0,2.0,NaN,NaN,NaN
1,10001334,546.325998,2,1,2,1,1,3,1.0,3.0,...,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
2,10001751,342.195549,1,3,5,2,1,1,3.0,14.0,...,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
3,10002171,335.358722,2,4,3,2,2,1,3.0,NaN,...,NaN,NaN,0.0,0.0,4.0,1.0,1.0,1.0,NaN,NaN
4,10003462,1941.603587,2,3,4,2,1,2,1.0,18.0,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0


In [62]:
nsduh_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58633 entries, 0 to 58632
Data columns (total 21 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   QUESTID2     58633 non-null  int64  
 1   ANALWT2_C    58633 non-null  float64
 2   COUTYP4      58633 non-null  int64  
 3   AGE3         58633 non-null  int64  
 4   HEALTH       58633 non-null  int64  
 5   IRSEX        58633 non-null  int64  
 6   CATAG6       58633 non-null  int64  
 7   NEWRACE2     58633 non-null  int64  
 8   POVERTY3     58497 non-null  float64
 9   YGADTOTSC    11334 non-null  float64
 10  YGADSEVSYMP  11334 non-null  float64
 11  YUSUITHKYR   11299 non-null  float64
 12  YUSUIPLNYR   11299 non-null  float64
 13  IRSUICTHNK   47299 non-null  float64
 14  IRSUIPLANYR  47299 non-null  float64
 15  AGADTOTSC    47299 non-null  float64
 16  AGADSEVSYMP  47299 non-null  float64
 17  AMDEYR       45659 non-null  float64
 18  AMDETXRX     9293 non-null   float64
 19  YMDE

## Dataset 2: CDC suicide death data

In [63]:

cdc_raw = {}
cdc_raw["trend"] = pd.read_csv("data-table_Rates-Gen_2026 (2).csv", dtype=str)
cdc_raw["age"] = pd.read_csv("data-table_RatesByAge.csv", dtype=str)
cdc_raw["sex"] = pd.read_csv("data-table_RatesBySex.csv", dtype=str)
cdc_raw["race"] = pd.read_csv("data-table_rates-by-race-ethnicity.csv", dtype=str)

for key in cdc_raw:
    print(key, cdc_raw[key].shape)

trend (24, 3)
age (16, 2)
sex (2, 2)
race (7, 2)


In [64]:
cdc_raw["trend"].head()

,Year,"Rate per 100,000",Count
0,2001,10.7,"30,622"
1,2002,11,"31,655"
2,2003,10.8,"31,484"
3,2004,11,"32,439"
4,2005,10.9,"32,637"


In [65]:
cdc_raw["age"]

,Age range,"Crude rate per 100,000 population"
0,85+,21.43
1,80-84,21.67
2,75-79,17.66
3,70-74,15.86
4,65-69,14.74
5,60-64,17.95
6,55-59,19.28
7,50-54,19.7
8,45-49,17.95
9,40-44,18.82


In [66]:
cdc_raw["sex"]

,Sex,"Age-adjusted rate per 100,000 population"
0,Males,22.23
1,Females,5.58


In [67]:
cdc_raw["race"]

,Race/ethnicity,"Age-adjusted rate per 100,000 population"
0,Non-Hispanic American Indian/Alaska Native,22.45
1,Non-Hispanic White,17.16
2,Non-Hispanic Native Hawaiian/Other Pacific Isl...,13.67
3,Non-Hispanic Multiracial,10.1
4,Non-Hispanic Black,8.66
5,Hispanic,7.96
6,Non-Hispanic Asian,6.43


In [68]:
for key in cdc_raw:
    print(key, list(cdc_raw[key].columns))

trend ['Year', 'Rate per 100,000', 'Count']
age ['Age range', 'Crude rate per 100,000 population']
sex ['Sex', 'Age-adjusted rate per 100,000 population']
race ['Race/ethnicity', 'Age-adjusted rate per 100,000 population']


# Part 2: Cleaning

## Cleaning Dataset 1: NSDUH

In [69]:
df = nsduh_raw.copy()
print("Starting rows:", len(df))

Starting rows: 58633


### Step 1: Remove duplicates

In [70]:
print("Duplicates:", df.duplicated(subset=["QUESTID2"]).sum())
df = df.drop_duplicates(subset=["QUESTID2"])
print("Rows:", len(df))

Duplicates: 0
Rows: 58633


### Step 2: Keep ages 12 to 25

In [71]:
df = df[df["CATAG6"] <= 2].copy()
print("Rows:", len(df))
df["CATAG6"].value_counts()

Rows: 25470


,count
CATAG6,
2,14136
1,11334


### Step 3: Rename columns

In [72]:
df = df.rename(columns={
    "QUESTID2": "person_id",
    "ANALWT2_C": "weight",
    "CATAG6": "age_cat",
    "AGE3": "age_code",
    "IRSEX": "sex_code",
    "NEWRACE2": "race_code",
    "COUTYP4": "county_code",
    "POVERTY3": "poverty_code",
    "HEALTH": "health_code",
    "YMDEYR": "teen_depression",
    "AMDEYR": "adult_depression",
    "YUSUITHKYR": "teen_thoughts",
    "IRSUICTHNK": "adult_thoughts",
    "YUSUIPLNYR": "teen_plan",
    "IRSUIPLANYR": "adult_plan",
    "YGADSEVSYMP": "teen_anxiety_level",
    "AGADSEVSYMP": "adult_anxiety_level",
    "YGADTOTSC": "teen_anxiety_score",
    "AGADTOTSC": "adult_anxiety_score",
    "YMDETXRX": "teen_treatment",
    "AMDETXRX": "adult_treatment",
})
df.columns

Index(['person_id', 'weight', 'county_code', 'age_code', 'health_code',
       'sex_code', 'age_cat', 'race_code', 'poverty_code',
       'teen_anxiety_score', 'teen_anxiety_level', 'teen_thoughts',
       'teen_plan', 'adult_thoughts', 'adult_plan', 'adult_anxiety_score',
       'adult_anxiety_level', 'adult_depression', 'adult_treatment',
       'teen_depression', 'teen_treatment'],
      dtype='object')

### Step 4: Change "not sure" and "refused" codes to NaN

In [73]:

df["teen_thoughts"] = df["teen_thoughts"].replace({3: np.nan, 4: np.nan})
df["teen_plan"] = df["teen_plan"].replace({3: np.nan, 4: np.nan})

df["health_code"] = df["health_code"].replace({94: np.nan, 97: np.nan})

print("Teen thoughts:", df["teen_thoughts"].value_counts().to_dict())
print("Teen plan:", df["teen_plan"].value_counts().to_dict())
print("Health:", df["health_code"].value_counts().to_dict())

Teen thoughts: {2.0: 8490, 1.0: 1220}
Teen plan: {2.0: 9709, 1.0: 596}
Health: {2.0: 9373, 3.0: 6978, 1.0: 6854, 4.0: 2029, 5.0: 216}


### Step 5: Combine teen and adult columns (1 = yes, 0 = no)

In [74]:
teen = df["age_cat"] == 1

df["depression"] = np.where(teen, df["teen_depression"], df["adult_depression"])
df["depression"] = df["depression"].map({1: 1, 2: 0})

df["teen_thoughts"] = df["teen_thoughts"].map({1: 1, 2: 0})
df["suicidal_thoughts"] = np.where(teen, df["teen_thoughts"], df["adult_thoughts"])

df["teen_plan"] = df["teen_plan"].map({1: 1, 2: 0})
df["suicide_plan"] = np.where(teen, df["teen_plan"], df["adult_plan"])

df["anxiety_level"] = np.where(teen, df["teen_anxiety_level"], df["adult_anxiety_level"])
df["anxiety_score"] = np.where(teen, df["teen_anxiety_score"], df["adult_anxiety_score"])

df["got_treatment"] = np.where(teen, df["teen_treatment"], df["adult_treatment"])

df[["age_cat", "depression", "suicidal_thoughts", "suicide_plan",
    "anxiety_level", "anxiety_score", "got_treatment"]].head(10)

,age_cat,depression,suicidal_thoughts,suicide_plan,anxiety_level,anxiety_score,got_treatment
1,1,0.0,0.0,0.0,1.0,3.0,NaN
2,1,0.0,1.0,0.0,3.0,14.0,NaN
3,2,1.0,0.0,0.0,1.0,4.0,1.0
4,1,1.0,1.0,1.0,4.0,18.0,0.0
5,2,0.0,0.0,0.0,1.0,0.0,NaN
7,2,0.0,0.0,0.0,1.0,0.0,NaN
12,2,0.0,0.0,0.0,4.0,20.0,NaN
14,2,0.0,0.0,0.0,1.0,1.0,NaN
17,2,NaN,0.0,0.0,1.0,0.0,NaN
19,2,0.0,0.0,0.0,1.0,0.0,NaN


### Step 6: Make the anxiety yes/no column

In [75]:

df["anxiety"] = df["anxiety_level"].map({1: 0, 2: 0, 3: 1, 4: 1})
df["anxiety"].value_counts()

,count
anxiety,
0,21185
1,4285


### Step 7: Treatment only for people with past-year depression

In [76]:
df["got_treatment"] = np.where(df["depression"] == 1, df["got_treatment"], np.nan)
print("Treatment answers:", df["got_treatment"].count())
print("People with depression:", (df["depression"] == 1).sum())

Treatment answers: 3984
People with depression: 4005


### Step 8: Drop the old teen and adult columns

In [23]:
df = df.drop(columns=["teen_depression", "adult_depression", "teen_thoughts", "adult_thoughts",
                      "teen_plan", "adult_plan", "teen_anxiety_level", "adult_anxiety_level",
                      "teen_anxiety_score", "adult_anxiety_score", "teen_treatment", "adult_treatment"])
print("Columns:", df.shape[1])

Columns: 16


### Step 9: Change codes to labels

In [77]:
df["age_group"] = df["age_cat"].map({1: "Ages 12-17", 2: "Ages 18-25"})
df["age_range"] = df["age_code"].map({1: "12-13", 2: "14-15", 3: "16-17",
                                      4: "18-20", 5: "21-23", 6: "24-25"})
df["sex"] = df["sex_code"].map({1: "Male", 2: "Female"})
df["race_ethnicity"] = df["race_code"].map({1: "White", 2: "Black", 3: "American Indian/Alaska Native",
                                            4: "Native Hawaiian/Pacific Islander", 5: "Asian",
                                            6: "More than one race", 7: "Hispanic"})
df["county_type"] = df["county_code"].map({1: "Large metro", 2: "Small metro", 3: "Nonmetro (rural)"})
df["family_income"] = df["poverty_code"].map({1: "Below poverty line", 2: "Up to 2x poverty line",
                                              3: "More than 2x poverty line"})
df["overall_health"] = df["health_code"].map({1: "Excellent", 2: "Very good", 3: "Good",
                                              4: "Fair", 5: "Poor"})
df["anxiety_level_name"] = df["anxiety_level"].map({1: "Minimal (0-4)", 2: "Mild (5-9)",
                                                    3: "Moderate (10-14)", 4: "Severe (15-21)"})
df[["age_group", "age_range", "sex", "race_ethnicity", "county_type", "family_income", "overall_health"]].head()

,age_group,age_range,sex,race_ethnicity,county_type,family_income,overall_health
1,Ages 12-17,12-13,Male,American Indian/Alaska Native,Small metro,Below poverty line,Very good
2,Ages 12-17,16-17,Female,White,Large metro,More than 2x poverty line,Poor
3,Ages 18-25,18-20,Female,White,Small metro,More than 2x poverty line,Good
4,Ages 12-17,16-17,Female,Black,Small metro,Below poverty line,Fair
5,Ages 18-25,24-25,Female,White,Large metro,More than 2x poverty line,Very good


### Step 10: Fix spaces and spelling

In [78]:
text_columns = ["age_group", "age_range", "sex", "race_ethnicity", "county_type",
                "family_income", "overall_health", "anxiety_level_name"]
for col in text_columns:
    df[col] = df[col].str.strip()
    print(col, ":", df[col].unique())

age_group : ['Ages 12-17' 'Ages 18-25']
age_range : ['12-13' '16-17' '18-20' '24-25' '21-23' '14-15']
sex : ['Male' 'Female']
race_ethnicity : ['American Indian/Alaska Native' 'White' 'Black' 'Hispanic' 'Asian'
 'More than one race' 'Native Hawaiian/Pacific Islander']
county_type : ['Small metro' 'Large metro' 'Nonmetro (rural)']
family_income : ['Below poverty line' 'More than 2x poverty line' 'Up to 2x poverty line'
 nan]
overall_health : ['Very good' 'Poor' 'Good' 'Fair' 'Excellent' nan]
anxiety_level_name : ['Minimal (0-4)' 'Moderate (10-14)' 'Severe (15-21)' 'Mild (5-9)']


In [79]:

pairs = [("age_cat", "age_group"), ("age_code", "age_range"), ("sex_code", "sex"),
         ("race_code", "race_ethnicity"), ("county_code", "county_type"),
         ("poverty_code", "family_income"), ("health_code", "overall_health"),
         ("anxiety_level", "anxiety_level_name")]
for code_col, label_col in pairs:
    print(code_col, "->", label_col, "| match:", df[code_col].count() == df[label_col].count())

age_cat -> age_group | match: True
age_code -> age_range | match: True
sex_code -> sex | match: True
race_code -> race_ethnicity | match: True
county_code -> county_type | match: True
poverty_code -> family_income | match: True
health_code -> overall_health | match: True
anxiety_level -> anxiety_level_name | match: True


### Step 11: Make the mental health group column

In [80]:
def mental_health_group(row):
    if row["depression"] == 1 and row["anxiety"] == 1:
        return "Both"
    elif row["depression"] == 1 and row["anxiety"] == 0:
        return "Depression only"
    elif row["depression"] == 0 and row["anxiety"] == 1:
        return "Anxiety only"
    elif row["depression"] == 0 and row["anxiety"] == 0:
        return "Neither"
    else:
        return np.nan

df["mh_group"] = df.apply(mental_health_group, axis=1)
df["mh_group"].value_counts(dropna=False)

,count
mh_group,
Neither,18644
Both,2184
Anxiety only,1903
Depression only,1821
NaN,918


### Step 12: Check the values

In [81]:

for col in ["depression", "suicidal_thoughts", "suicide_plan", "anxiety", "got_treatment"]:
    print(col, ":", df[col].dropna().unique())

depression : [0. 1.]
suicidal_thoughts : [0. 1.]
suicide_plan : [0. 1.]
anxiety : [0 1]
got_treatment : [1. 0.]


In [82]:

df[["weight", "anxiety_score", "anxiety_level"]].describe().round(2)

,weight,anxiety_score,anxiety_level
count,25470.00,25470.00,25470.00
mean,2390.95,4.57,1.63
std,3142.29,5.40,0.92
min,1.56,0.00,1.00
25%,464.05,0.00,1.00
50%,1244.72,3.00,1.00
75%,3047.27,7.00,2.00
max,46958.38,21.00,4.00


In [83]:

df.groupby("anxiety_level")["anxiety_score"].agg(["min", "max"])

,min,max
anxiety_level,,
1.0,0.0,4.0
2.0,5.0,9.0
3.0,10.0,14.0
4.0,15.0,21.0


In [31]:

print(len(df[(df["suicide_plan"] == 1) & (df["suicidal_thoughts"] == 0)]))

22


### Step 13: Missing values (1,000 rule)

In [84]:
missing_count = len(df) - df.count()
missing_table = pd.DataFrame({"missing": missing_count,
                              "percent": (missing_count / len(df) * 100).round(1)})
missing_table = missing_table[missing_table["missing"] > 0].sort_values("missing", ascending=False)

missing_table.to_csv(CLEAN + "nsduh_missing_values_before_dropna.csv")
missing_table

,missing,percent
teen_treatment,23000,90.3
adult_treatment,22017,86.4
got_treatment,21486,84.4
teen_thoughts,15760,61.9
teen_plan,15165,59.5
teen_depression,14553,57.1
teen_anxiety_level,14136,55.5
teen_anxiety_score,14136,55.5
adult_depression,11835,46.5
adult_anxiety_level,11334,44.5


In [85]:
keep_rows = missing_table[missing_table["missing"] > 1000].index.tolist()
drop_rows = missing_table[missing_table["missing"] <= 1000].index.tolist()
print("More than 1,000 missing (keep):", keep_rows)
print("1,000 or fewer missing (dropna):", drop_rows)

More than 1,000 missing (keep): ['teen_treatment', 'adult_treatment', 'got_treatment', 'teen_thoughts', 'teen_plan', 'teen_depression', 'teen_anxiety_level', 'teen_anxiety_score', 'adult_depression', 'adult_anxiety_level', 'adult_thoughts', 'adult_anxiety_score', 'adult_plan', 'suicidal_thoughts', 'suicide_plan']
1,000 or fewer missing (dropna): ['depression', 'mh_group', 'family_income', 'poverty_code', 'health_code', 'overall_health']


In [86]:
rows_before = len(df)
df = df.dropna(subset=drop_rows)
print("Rows before:", rows_before)
print("Rows after:", len(df))
print("Dropped:", rows_before - len(df), "| percent:", round((rows_before - len(df)) / rows_before * 100, 1))

Rows before: 25470
Rows after: 24406
Dropped: 1064 | percent: 4.2


In [87]:

(len(df) - df.count())[(len(df) - df.count()) > 0]

,0
teen_anxiety_score,13495
teen_anxiety_level,13495
teen_thoughts,14904
teen_plan,14341
adult_thoughts,10911
adult_plan,10911
adult_anxiety_score,10911
adult_anxiety_level,10911
adult_depression,10911
adult_treatment,21012


### Step 14: Set data types

In [88]:
whole_numbers = ["person_id", "age_cat", "age_code", "sex_code", "race_code", "county_code",
                 "poverty_code", "health_code", "depression", "anxiety_level", "anxiety_score", "anxiety"]
for col in whole_numbers:
    df[col] = df[col].astype(int)

for col in text_columns + ["mh_group"]:
    df[col] = df[col].astype("category")

df.dtypes

,0
person_id,int64
weight,float64
county_code,int64
age_code,int64
health_code,int64
sex_code,int64
age_cat,int64
race_code,int64
poverty_code,int64
teen_anxiety_score,float64


### Step 15: Order the columns and save

In [89]:
df = df[["person_id", "weight",
         "age_cat", "age_group", "age_code", "age_range",
         "sex_code", "sex", "race_code", "race_ethnicity",
         "county_code", "county_type", "poverty_code", "family_income",
         "health_code", "overall_health",
         "depression", "anxiety_level", "anxiety_level_name", "anxiety_score", "anxiety",
         "mh_group", "suicidal_thoughts", "suicide_plan", "got_treatment"]]

df.to_csv(CLEAN + "nsduh_2024_clean.csv", index=False)
print("Saved:", df.shape)
df.head()

Saved: (24406, 25)


,person_id,weight,age_cat,age_group,age_code,age_range,sex_code,sex,race_code,race_ethnicity,...,overall_health,depression,anxiety_level,anxiety_level_name,anxiety_score,anxiety,mh_group,suicidal_thoughts,suicide_plan,got_treatment
1,10001334,546.325998,1,Ages 12-17,1,12-13,1,Male,3,American Indian/Alaska Native,...,Very good,0,1,Minimal (0-4),3,0,Neither,0.0,0.0,NaN
2,10001751,342.195549,1,Ages 12-17,3,16-17,2,Female,1,White,...,Poor,0,3,Moderate (10-14),14,1,Anxiety only,1.0,0.0,NaN
3,10002171,335.358722,2,Ages 18-25,4,18-20,2,Female,1,White,...,Good,1,1,Minimal (0-4),4,0,Depression only,0.0,0.0,1.0
4,10003462,1941.603587,1,Ages 12-17,3,16-17,2,Female,2,Black,...,Fair,1,4,Severe (15-21),18,1,Both,1.0,1.0,0.0
5,10004278,7372.604360,2,Ages 18-25,6,24-25,2,Female,1,White,...,Very good,0,1,Minimal (0-4),0,0,Neither,0.0,0.0,NaN


### NSDUH data dictionary

In [90]:
dictionary = pd.DataFrame([
    ["person_id", "Person ID (QUESTID2)"],
    ["weight", "Survey weight (ANALWT2_C)"],
    ["age_cat / age_group", "1 = Ages 12-17, 2 = Ages 18-25"],
    ["age_code / age_range", "1 = 12-13, 2 = 14-15, 3 = 16-17, 4 = 18-20, 5 = 21-23, 6 = 24-25"],
    ["sex_code / sex", "1 = Male, 2 = Female"],
    ["race_code / race_ethnicity", "Race/ethnicity, 7 groups"],
    ["county_code / county_type", "1 = Large metro, 2 = Small metro, 3 = Nonmetro"],
    ["poverty_code / family_income", "1 = Below poverty, 2 = Up to 2x poverty, 3 = More than 2x poverty"],
    ["health_code / overall_health", "1 = Excellent to 5 = Poor"],
    ["depression", "Depression in past year: 1 = yes, 0 = no"],
    ["anxiety_level / anxiety_level_name", "1 = Minimal, 2 = Mild, 3 = Moderate, 4 = Severe"],
    ["anxiety_score", "GAD-7 score, 0 to 21"],
    ["anxiety", "1 = score 10 or more, 0 = under 10"],
    ["mh_group", "Neither / Anxiety only / Depression only / Both"],
    ["suicidal_thoughts", "Serious suicidal thoughts past year: 1 = yes, 0 = no"],
    ["suicide_plan", "Suicide plan past year: 1 = yes, 0 = no"],
    ["got_treatment", "Treatment for depression past year (only people with depression)"],
], columns=["column", "meaning"])
dictionary

,column,meaning
0,person_id,Person ID (QUESTID2)
1,weight,Survey weight (ANALWT2_C)
2,age_cat / age_group,"1 = Ages 12-17, 2 = Ages 18-25"
3,age_code / age_range,"1 = 12-13, 2 = 14-15, 3 = 16-17, 4 = 18-20, 5 ..."
4,sex_code / sex,"1 = Male, 2 = Female"
5,race_code / race_ethnicity,"Race/ethnicity, 7 groups"
6,county_code / county_type,"1 = Large metro, 2 = Small metro, 3 = Nonmetro"
7,poverty_code / family_income,"1 = Below poverty, 2 = Up to 2x poverty, 3 = M..."
8,health_code / overall_health,1 = Excellent to 5 = Poor
9,depression,"Depression in past year: 1 = yes, 0 = no"


## Cleaning Dataset 2: CDC

### Step 1: Clean the column names

In [91]:
cdc = {}
for key in cdc_raw:
    table = cdc_raw[key].copy()
    table.columns = table.columns.str.replace("\ufeff", "").str.strip()
    cdc[key] = table
    print(key, list(table.columns))

trend ['Year', 'Rate per 100,000', 'Count']
age ['Age range', 'Crude rate per 100,000 population']
sex ['Sex', 'Age-adjusted rate per 100,000 population']
race ['Race/ethnicity', 'Age-adjusted rate per 100,000 population']


### Step 2: Rename columns and add the rate type

In [92]:
rename_cdc = {
    "Year": "year",
    "Rate per 100,000": "rate",
    "Count": "deaths",
    "Age range": "age_group",
    "Crude rate per 100,000 population": "rate",
    "Sex": "sex",
    "Age-adjusted rate per 100,000 population": "rate",
    "Race/ethnicity": "race_ethnicity",
}
rate_types = {"trend": "age-adjusted", "age": "crude", "sex": "age-adjusted", "race": "age-adjusted"}

for key in cdc:
    cdc[key] = cdc[key].rename(columns=rename_cdc)
    cdc[key]["rate_type"] = rate_types[key]
    print(key, list(cdc[key].columns))

trend ['year', 'rate', 'deaths', 'rate_type']
age ['age_group', 'rate', 'rate_type']
sex ['sex', 'rate', 'rate_type']
race ['race_ethnicity', 'rate', 'rate_type']


### Step 3: Fix the text

In [93]:
for key in cdc:
    for col in cdc[key].columns:
        cdc[key][col] = cdc[key][col].str.strip()

cdc["sex"]["sex"] = cdc["sex"]["sex"].replace({"Males": "Male", "Females": "Female"})

print(cdc["sex"]["sex"].unique())
print(cdc["age"]["age_group"].unique())
print(cdc["race"]["race_ethnicity"].unique())

['Male' 'Female']
['85+' '80-84' '75-79' '70-74' '65-69' '60-64' '55-59' '50-54' '45-49'
 '40-44' '35-39' '30-34' '25-29' '20-24' '15-19' '10-14']
['Non-Hispanic American Indian/Alaska Native' 'Non-Hispanic White'
 'Non-Hispanic Native Hawaiian/Other Pacific Islander'
 'Non-Hispanic Multiracial' 'Non-Hispanic Black' 'Hispanic'
 'Non-Hispanic Asian']


### Step 4: Change text to numbers

In [94]:
number_columns = ["year", "rate", "deaths"]

for key in cdc:
    for col in number_columns:
        if col in cdc[key].columns:
            cdc[key][col] = cdc[key][col].str.replace(",", "")            # "30,622" -> "30622"
            cdc[key][col] = pd.to_numeric(cdc[key][col], errors="coerce")
    print(key)
    print(cdc[key].dtypes, "\n")

trend
year           int64
rate         float64
deaths         int64
rate_type     object
dtype: object 

age
age_group     object
rate         float64
rate_type     object
dtype: object 

sex
sex           object
rate         float64
rate_type     object
dtype: object 

race
race_ethnicity     object
rate              float64
rate_type          object
dtype: object 



### Step 5: Remove duplicates

In [95]:
group_columns = {"trend": "year", "age": "age_group", "sex": "sex", "race": "race_ethnicity"}

for key in cdc:
    print(key, "duplicates:", cdc[key].duplicated(subset=[group_columns[key]]).sum())
    cdc[key] = cdc[key].drop_duplicates(subset=[group_columns[key]])

trend duplicates: 0
age duplicates: 0
sex duplicates: 0
race duplicates: 0


### Step 6: Check the values

In [96]:
years = sorted(cdc["trend"]["year"].tolist())
print("Years:", years[0], "to", years[-1], "| count:", len(years))
print("All years 2001-2024:", years == list(range(2001, 2025)))
print("Deaths 0 or less:", (cdc["trend"]["deaths"] <= 0).sum())

for key in cdc:
    print(key, "| min rate:", cdc[key]["rate"].min(), "| max rate:", cdc[key]["rate"].max(),
          "| negative:", (cdc[key]["rate"] < 0).sum())

print("Sex groups:", cdc["sex"]["sex"].nunique())
print("Race groups:", cdc["race"]["race_ethnicity"].nunique())
print("Age groups:", cdc["age"]["age_group"].nunique())

Years: 2001 to 2024 | count: 24
All years 2001-2024: True
Deaths 0 or less: 0
trend | min rate: 10.7 | max rate: 14.2 | negative: 0
age | min rate: 2.33 | max rate: 21.67 | negative: 0
sex | min rate: 5.58 | max rate: 22.23 | negative: 0
race | min rate: 6.43 | max rate: 22.45 | negative: 0
Sex groups: 2
Race groups: 7
Age groups: 16


### Step 7: Missing values (1,000 rule)

In [97]:
cdc_missing_tables = []
for key in cdc:
    missing = len(cdc[key]) - cdc[key].count()
    cdc_missing_tables.append(pd.DataFrame({"file": key, "column": missing.index, "missing": missing.values}))

cdc_missing = pd.concat(cdc_missing_tables)
cdc_missing.to_csv(CLEAN + "cdc_missing_values_before_dropna.csv", index=False)
print("Total missing:", cdc_missing["missing"].sum())

Total missing: 0


In [98]:
for key in cdc:
    missing = len(cdc[key]) - cdc[key].count()
    drop_rows = missing[(missing > 0) & (missing <= 1000)].index.tolist()
    cdc[key] = cdc[key].dropna(subset=drop_rows)
    print(key, "| dropna columns:", drop_rows, "| rows:", len(cdc[key]))

trend | dropna columns: [] | rows: 24
age | dropna columns: [] | rows: 16
sex | dropna columns: [] | rows: 2
race | dropna columns: [] | rows: 7


### Step 8: Add helper columns

In [99]:

age_start = []
for label in cdc["age"]["age_group"]:
    age_start.append(int(label.replace("+", "").split("-")[0]))
cdc["age"]["age_start"] = age_start

cdc["age"]["young_people"] = np.where(cdc["age"]["age_group"].isin(["10-14", "15-19", "20-24"]), 1, 0)

cdc["race"]["race_short"] = cdc["race"]["race_ethnicity"].str.replace("Non-Hispanic ", "")

cdc["age"].sort_values("age_start")

,age_group,rate,rate_type,age_start,young_people
15,10-14,2.33,crude,10,1
14,15-19,9.41,crude,15,1
13,20-24,16.99,crude,20,1
12,25-29,17.04,crude,25,0
11,30-34,17.37,crude,30,0
10,35-39,18.58,crude,35,0
9,40-44,18.82,crude,40,0
8,45-49,17.95,crude,45,0
7,50-54,19.70,crude,50,0
6,55-59,19.28,crude,55,0


### Step 9: Set data types, sort, and save

In [48]:
cdc["trend"]["year"] = cdc["trend"]["year"].astype(int)
cdc["trend"]["deaths"] = cdc["trend"]["deaths"].astype(int)
for key in cdc:
    cdc[key]["rate"] = cdc[key]["rate"].astype(float)

cdc["trend"] = cdc["trend"].sort_values("year")
cdc["age"] = cdc["age"].sort_values("age_start")
cdc["sex"] = cdc["sex"].sort_values("rate", ascending=False)
cdc["race"] = cdc["race"].sort_values("rate", ascending=False)

cdc["trend"].to_csv(CLEAN + "cdc_suicide_trend_2001_2024_clean.csv", index=False)
cdc["age"].to_csv(CLEAN + "cdc_suicide_by_age_clean.csv", index=False)
cdc["sex"].to_csv(CLEAN + "cdc_suicide_by_sex_clean.csv", index=False)
cdc["race"].to_csv(CLEAN + "cdc_suicide_by_race_ethnicity_clean.csv", index=False)

for key in cdc:
    print(key, cdc[key].shape)

trend (24, 4)
age (16, 5)
sex (2, 3)
race (7, 4)


In [100]:
cdc["trend"].head()

,year,rate,deaths,rate_type
0,2001,10.7,30622,age-adjusted
1,2002,11.0,31655,age-adjusted
2,2003,10.8,31484,age-adjusted
3,2004,11.0,32439,age-adjusted
4,2005,10.9,32637,age-adjusted


In [50]:
cdc["age"]

,age_group,rate,rate_type,age_start,young_people
15,10-14,2.33,crude,10,1
14,15-19,9.41,crude,15,1
13,20-24,16.99,crude,20,1
12,25-29,17.04,crude,25,0
11,30-34,17.37,crude,30,0
10,35-39,18.58,crude,35,0
9,40-44,18.82,crude,40,0
8,45-49,17.95,crude,45,0
7,50-54,19.70,crude,50,0
6,55-59,19.28,crude,55,0


In [101]:
cdc["sex"]

,sex,rate,rate_type
0,Male,22.23,age-adjusted
1,Female,5.58,age-adjusted


In [102]:
cdc["race"]

,race_ethnicity,rate,rate_type,race_short
0,Non-Hispanic American Indian/Alaska Native,22.45,age-adjusted,American Indian/Alaska Native
1,Non-Hispanic White,17.16,age-adjusted,White
2,Non-Hispanic Native Hawaiian/Other Pacific Isl...,13.67,age-adjusted,Native Hawaiian/Other Pacific Islander
3,Non-Hispanic Multiracial,10.10,age-adjusted,Multiracial
4,Non-Hispanic Black,8.66,age-adjusted,Black
5,Hispanic,7.96,age-adjusted,Hispanic
6,Non-Hispanic Asian,6.43,age-adjusted,Asian


### CDC data dictionary

In [103]:
cdc_dictionary = pd.DataFrame([
    ["year", "trend", "Year, 2001 to 2024"],
    ["deaths", "trend", "Number of suicide deaths"],
    ["age_group", "age", "5-year age group"],
    ["age_start", "age", "First age in the group (for sorting)"],
    ["young_people", "age", "1 = ages 10-24, 0 = older"],
    ["sex", "sex", "Male or Female"],
    ["race_ethnicity", "race", "Race/ethnicity as the CDC wrote it"],
    ["race_short", "race", "Shorter label (all groups except Hispanic are non-Hispanic)"],
    ["rate", "all", "Suicide deaths per 100,000 people"],
    ["rate_type", "all", "crude or age-adjusted"],
], columns=["column", "file", "meaning"])
cdc_dictionary

,column,file,meaning
0,year,trend,"Year, 2001 to 2024"
1,deaths,trend,Number of suicide deaths
2,age_group,age,5-year age group
3,age_start,age,First age in the group (for sorting)
4,young_people,age,"1 = ages 10-24, 0 = older"
5,sex,sex,Male or Female
6,race_ethnicity,race,Race/ethnicity as the CDC wrote it
7,race_short,race,Shorter label (all groups except Hispanic are ...
8,rate,all,"Suicide deaths per 100,000 people"
9,rate_type,all,crude or age-adjusted


## Final check of my files I cleaned

In [104]:
for name in sorted(os.listdir(CLEAN)):
    clean = pd.read_csv(CLEAN + name)
    print(name, "| rows:", clean.shape[0], "| columns:", clean.shape[1],
          "| duplicates:", clean.duplicated().sum())

cdc_missing_values_before_dropna.csv | rows: 13 | columns: 3 | duplicates: 0
cdc_suicide_by_age_clean.csv | rows: 16 | columns: 5 | duplicates: 0
cdc_suicide_by_race_ethnicity_clean.csv | rows: 7 | columns: 4 | duplicates: 0
cdc_suicide_by_sex_clean.csv | rows: 2 | columns: 3 | duplicates: 0
cdc_suicide_trend_2001_2024_clean.csv | rows: 24 | columns: 4 | duplicates: 0
nsduh_2024_clean.csv | rows: 24406 | columns: 25 | duplicates: 0
nsduh_missing_values_before_dropna.csv | rows: 21 | columns: 3 | duplicates: 0


# Part 3: Downloading my cleaned files

In [56]:
from google.colab import files

files.download(CLEAN + "nsduh_2024_clean.csv")
files.download(CLEAN + "cdc_suicide_trend_2001_2024_clean.csv")
files.download(CLEAN + "cdc_suicide_by_age_clean.csv")
files.download(CLEAN + "cdc_suicide_by_sex_clean.csv")
files.download(CLEAN + "cdc_suicide_by_race_ethnicity_clean.csv")
files.download(CLEAN + "nsduh_missing_values_before_dropna.csv")
files.download(CLEAN + "cdc_missing_values_before_dropna.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>